In [ ]:
from torch import nn
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather')['A'].to_numpy()
data = data[~np.isnan(data)][:100]
plt.plot(data)

In [ ]:
convolution = nn.Conv1d(1, 1, 3)

In [ ]:
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather')
data

In [ ]:
from torch import nn
import torch
import pandas as pd
import numpy as np
import math

### Hyperparameters

batch_size = 32
n_stocks = 10
block_size = 20
n_eval = 200
eval_interval = 500
n_epochs = 1000
learning_rate = 1e-3
d_model = 64
c_in = 1 # Num Features
p_eval = 0.1

### Data
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather').to_numpy()
### Data Shape: (n_timesteps, n_stocks)

nan_counts = np.isnan(data).sum(axis=1)
invalidTimesteps = np.argwhere(nan_counts > data.shape[1] - n_stocks).squeeze()
for value in invalidTimesteps:
    invalidTimesteps = np.append(invalidTimesteps, np.arange(value-block_size+1, value))
invalidTimesteps = np.unique(invalidTimesteps)
data = np.delete(data, invalidTimesteps, axis=0)
split = int(data.shape[0]*(1-p_eval))
train_data = data[:split]
val_data = data[split:]
np.random.seed(0)
def get_batch(split='train'):
    data_split = train_data if split == 'train' else val_data
    batch_time_indices = np.random.randint(0, data_split.shape[0] - block_size - 1, size=batch_size)
    x_list = []
    y_list = []
    stocks_list = []
    for i in range(batch_size):
        time_i = batch_time_indices[i]
        data_i = data_split[time_i:time_i + block_size + 1]
        valid_stocks = np.where(~np.isnan(data_i).any(axis=0))[0]
        if len(valid_stocks) >= n_stocks:
            stocks_i = np.random.choice(valid_stocks, n_stocks, replace=False)
        else:
            stocks_i = np.random.choice(valid_stocks, n_stocks, replace=True)
        x_i = data_i[:-1, stocks_i].T
        y_i = data_i[1:, stocks_i].T 
        x_list.append(torch.tensor(x_i, dtype=torch.float32))
        y_list.append(torch.tensor(y_i, dtype=torch.float32))
        stocks_list.append(stocks_i)
    x = torch.stack(x_list) 
    y = torch.stack(y_list)
    stocks = np.array(stocks_list)

    return x, y, stocks

In [ ]:
%%timeit
get_batch()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather').to_numpy()
scaler.fit_transform(data)
print(data[0])


In [ ]:
from torch import nn
import torch
import pandas as pd
import numpy as np
import math
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from utils import find_max_non_outlier

batch_size = 32
block_size = 20
n_eval = 200
eval_interval = 1000
n_epochs = 5000
learning_rate = 1e-6
d_model = 64
p_eval = 0.1
scaler = StandardScaler()
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather').to_numpy()
T, S = data.shape
data = scaler.fit_transform(data.reshape(T*S, 1)).reshape(T, S)
split = int(data.shape[0]*(1-p_eval))
train_data = data[:split].T
val_data = data[split:].T
np.random.seed(0)
torch.manual_seed(0)
def get_batch(split='train'):
    data_split = train_data if split == 'train' else val_data
    stock = torch.randint(0, S, (batch_size,))
    x = []
    y= []
    for batch in range(batch_size):
        stockCurve = data_split[stock[batch]]
        stockCurve = stockCurve[~np.isnan(stockCurve)]
        while True:
            if len(stockCurve) < block_size + 9:
                stock[batch] = torch.randint(0, S, (1,))
                stockCurve = data_split[stock[batch]]
                stockCurve = stockCurve[~np.isnan(stockCurve)]
            else:
                break
        try:
            time = torch.randint(0, stockCurve.shape[0] - block_size - 8, (1,))
        except:
            raise Exception(stockCurve.shape[0] - block_size - 8)
        x.append(torch.tensor(stockCurve[time:time + block_size], dtype=torch.float32))
        curve = stockCurve[time + block_size + 1:time + block_size + 8]
        max_value, _ = find_max_non_outlier(torch.arange(7), curve)
        y.append(torch.tensor(max_value, dtype=torch.float32))
    x = torch.stack(x).to('cuda')
    y = torch.stack(y).to('cuda')
    return x, y, stock

In [ ]:
%%timeit
get_batch()

In [ ]:
import pandas as pd
import numpy as np
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather')
data

In [ ]:
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather')

data = data.to_numpy()
T, S = data.shape
data_filled = np.where(np.isnan(data), np.nanmean(data, axis=0), data).T
means = np.nanmean(data_filled, axis=0)
stds = np.std(data_filled, axis=0)
# data = (data_filled - means) / stds

In [ ]:
from torch import nn
import torch
import pandas as pd
import numpy as np
import math
import torch.nn.functional as F
from sklearn.preprocessing import RobustScaler
from tqdm import tqdm
from utils import find_max_non_outlier
### Hyperparameters

batch_size = 32
block_size = 20
n_eval = 100
eval_interval = 1000
n_epochs = 5000
learning_rate = 1e-4
d_model = 64
p_eval = 0.1

### Data

scaler = RobustScaler()
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather').to_numpy()
T, S = data.shape
data = scaler.fit_transform(data.reshape(T*S, 1)).reshape(T, S)
data

In [ ]:
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather').to_numpy()

eps = 1e-8
data = np.where(data <= 0, eps, data)
log_returns = np.log(data[1:] / data[:-1])
log_returns = np.nan_to_num(log_returns, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
log_returns

In [ ]:
from sklearn.preprocessing import StandardScaler
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather').to_numpy()
active_mask = ~np.isnan(data)
data_filled = pd.DataFrame(data).ffill().to_numpy()
log_prices = np.log(data_filled)
log_returns = np.diff(log_prices, axis=0)  # Shape: (T-1, S)
active_mask = active_mask[1:] & active_mask[:-1]  # Shape: (T-1, S)
log_returns = np.nan_to_num(log_returns, nan=0.0, posinf=0.0, neginf=0.0)
scaler = StandardScaler()
log_returns_flat = log_returns.reshape(-1, 1)
log_returns_scaled = scaler.fit_transform(log_returns_flat)
log_returns = log_returns_scaled.reshape(log_returns.shape)
T, S = log_returns.shape
split = int(T * (1 - p_eval))
train_data = log_returns[:split]
train_active_mask = active_mask[:split]
val_data = log_returns[split:]
val_active_mask = active_mask[split:]

In [ ]:
def get_batch(split='train'):
    if split == 'train':
        data_split = train_data  # Shape: (T_train, S)
        active_mask_split = train_active_mask  # Shape: (T_train, S)
    else:
        data_split = val_data  # Shape: (T_val, S)
        active_mask_split = val_active_mask  # Shape: (T_val, S)
    batch_x = []
    batch_y = []
    batch_stocks = []
    max_attempts = batch_size * 10
    attempts = 0
    while len(batch_x) < batch_size and attempts < max_attempts:
        stock_idx = np.random.randint(0, S)
        stock_log_returns = data_split[:, stock_idx]  # Shape: (T_split,)
        stock_active_mask = active_mask_split[:, stock_idx]  # Shape: (T_split,)
        active_indices = np.where(stock_active_mask)[0]
        if len(active_indices) < block_size + 7:
            attempts += 1
            continue  # Not enough data, try another stock
        possible_start_indices = active_indices[:-(block_size + 7) + 1]
        if len(possible_start_indices) == 0:
            attempts += 1
            continue  # No valid starting indices, try another stock
        start_idx = np.random.choice(possible_start_indices)
        input_indices = np.arange(start_idx, start_idx + block_size)
        future_indices = np.arange(start_idx + block_size, start_idx + block_size + 7)
        if not np.all(stock_active_mask[input_indices]) or not np.all(stock_active_mask[future_indices]):
            attempts += 1
            continue  # Inactive periods in window, try another selection
        x_seq = stock_log_returns[input_indices]  # Shape: (block_size,)
        price_0 = 1.0  # Arbitrary starting price
        cumulative_log_returns = np.cumsum(x_seq)
        prices = price_0 * np.exp(cumulative_log_returns)
        last_price_in_block = prices[-1]
        future_log_returns = stock_log_returns[future_indices]
        future_cumulative_log_returns = np.cumsum(future_log_returns)
        future_prices = last_price_in_block * np.exp(future_cumulative_log_returns)
        max_future_price = np.max(future_prices)
        target_log_return = np.log(max_future_price / last_price_in_block)
        batch_x.append(torch.tensor(x_seq, dtype=torch.float32))
        batch_y.append(torch.tensor(target_log_return, dtype=torch.float32))
        batch_stocks.append(stock_idx)
        attempts += 1
    
    if len(batch_x) == 0:
        raise ValueError("Unable to create a batch with valid data.")
    
    # Stack batches
    x_batch = torch.stack(batch_x).to('cuda')  # Shape: (batch_size, block_size)
    y_batch = torch.stack(batch_y).to('cuda')  # Shape: (batch_size,)
    stock_batch = torch.tensor(batch_stocks).to('cuda')
    
    return x_batch, y_batch, stock_batch

In [ ]:
get_batch()

In [ ]:
data = pd.read_feather('data/BigTestDataset/CLOSEDATAFINAL.feather').to_numpy()
data.shape[0]*.9